<a href="https://colab.research.google.com/github/ananthucherunthottam/gassafe-ai/blob/main/Copy_of_GasSafe_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/ananthucherunthottam/gassafe-ai.git
%cd /content/gassafe-ai
!python --version
!ls

Cloning into 'gassafe-ai'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 235 (delta 15), reused 1 (delta 1), pack-reused 192 (from 1)
Receiving objects: 100% (235/235), 128.64 KiB | 1.50 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/gassafe-ai
Python 3.13.16
app.py	   deploy	       ingest.py  README.md	    start.sh
config.py  docker-compose.yml  llm.py	  requirements.txt  tests
data	   Dockerfile	       rag.py	  start.ps1


In [ ]:
!pip -q install uv
!uv python install 3.11
!uv venv --python 3.11 /content/gassafe-ai/.venv
!/content/gassafe-ai/.venv/bin/python --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 76.3 MB/s eta 0:00:00
Installed Python 3.11.17 in 1.92s
 + cpython-3.11.17-linux-x86_64-gnu (python3.11)
Using CPython 3.11.17
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Python 3.11.17


In [ ]:
%cd /content/gassafe-ai

!uv pip install --python /content/gassafe-ai/.venv/bin/python -r requirements.txt

/content/gassafe-ai
Resolved 130 packages in 3.55s
Prepared 130 packages in 15m 44s
Installed 130 packages in 851ms
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiohttp-retry==2.9.1
 + aiosignal==1.4.0
 + altair==6.3.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + bcrypt==5.0.0
 + build==1.6.1
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + chromadb==1.5.9
 + click==8.5.0
 + cloudpickle==3.1.2
 + cuda-bindings==13.4.4
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + diskcache==5.6.3
 + durationpy==0.11
 + filelock==4.2.0
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + fsspec==2026.9.0
 + googleapis-common-protos==1.75.5
 + grpcio==1.84.0
 + h11==0.16.0
 + hf-xet==1.7.0
 + httpcore==1.0.9
 + httptools==0.9.0
 + httpx==0.28.1
 + huggingface-hub==0.36.2
 + idna==3.20
 + importlib-resources==7.1.0
 + itsdangerous==2.2.0
 + jinja2==3.1.6
 + joblib==1.6.0
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + kubernetes==37.0

In [ ]:
import subprocess

python = "/content/gassafe-ai/.venv/bin/python"

code = """
import streamlit
import chromadb
import sentence_transformers
import pypdf
import llama_cpp
import huggingface_hub

from config import PDF_DIR
from rag import ask_gassafe
from llm import load_model

print("PASS: Streamlit")
print("PASS: ChromaDB")
print("PASS: MiniLM dependencies")
print("PASS: PDF processing")
print("PASS: llama.cpp")
print("PASS: Hugging Face Hub")
print("PASS: GasSafe AI modules")
print("PDF files found:", len(list(PDF_DIR.glob("*.pdf"))))
"""

result = subprocess.run(
    [python, "-c", code],
    cwd="/content/gassafe-ai",
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-5000:])

PASS: Streamlit
PASS: ChromaDB
PASS: MiniLM dependencies
PASS: PDF processing
PASS: llama.cpp
PASS: Hugging Face Hub
PASS: GasSafe AI modules
PDF files found: 1



In [ ]:
import subprocess

result = subprocess.run(
    ["/content/gassafe-ai/.venv/bin/python", "ingest.py"],
    cwd="/content/gassafe-ai",
    capture_output=True,
    text=True,
    timeout=600
)

print("EXIT CODE:", result.returncode)
print("\nOUTPUT:")
print(result.stdout[-5000:])

if result.stderr:
    print("\nLOGS:")
    print(result.stderr[-3000:])

EXIT CODE: 0

OUTPUT:
Reading: GasSafe_AI_Synthetic_Pipeline_Manual.pdf
Preparing 10 document chunks...
Indexed 10/10 chunks
New index verified successfully.
Activating the new index...
GasSafe AI PDF indexing completed successfully.
Active collection: gas_safety_pdf
Database location: /content/gassafe-ai/chroma_db



In [ ]:
import subprocess
import time
import requests

process = subprocess.Popen(
    [
        "/content/gassafe-ai/.venv/bin/python",
        "-m", "streamlit", "run",
        "/content/gassafe-ai/app.py",
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false"
    ],
    cwd="/content/gassafe-ai",
    stdout=open("/content/gassafe-streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("Streamlit process:", process.pid)
print("Process running:", process.poll() is None)

try:
    response = requests.get(
        "http://localhost:8501/_stcore/health",
        timeout=10
    )
    print("Health status:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Connection error:", e)

print("\nRecent logs:")
print(open("/content/gassafe-streamlit.log").read()[-2000:])

Streamlit process: 8936
Process running: True
Health status: 200
Response: ok

Recent logs:
2026-10-15 20:58:16.953 
'server.enableCORS=false' tells Streamlit not to limit cross-origin access.
Streamlit sends the header 'Access-Control-Allow-Origin: *', except on the file
upload routes, which stay pinned to the app address and still require a valid
XSRF token.
Streamlit also accepts a WebSocket connection from any origin.
Streamlit does not set 'server.enableCORS' to 'true' for you.
'server.enableXsrfProtection=true' does not stop a cross-origin WebSocket
connection, because Streamlit does not need a valid XSRF token to open one.

To limit cross-origin access, do these steps:
  1. Set 'server.enableCORS=true'.
  2. Put the origins that you trust in 'server.corsAllowedOrigins'.

'server.allowedHosts' can also limit the hostnames that Streamlit accepts on a
WebSocket connection.



2026-10-15 20:58:17.650 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in yo

In [ ]:
import subprocess
import time
import re
import os
import requests

# Download Cloudflare Tunnel
subprocess.run(
    [
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/content/cloudflared"
    ],
    check=True
)

os.chmod("/content/cloudflared", 0o755)

# Start the tunnel
log_file = open("/content/cloudflare.log", "w")

tunnel = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url", "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for the temporary URL
url = None

for i in range(30):
    time.sleep(2)

    with open("/content/cloudflare.log") as f:
        logs = f.read()

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        logs
    )

    if match:
        url = match.group(0)
        break

if url:
    print("GasSafe AI URL:")
    print(url)
else:
    print("Tunnel URL not found.")
    print(logs[-3000:])

GasSafe AI URL:
https://working-burlington-entries-roman.trycloudflare.com


In [ ]:
import os
import subprocess

print("=== QWEN MODEL FILES ===")

result = subprocess.run(
    ["find", "/content/gassafe-ai/models",
     "-type", "f", "-name", "*.gguf"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "No GGUF model found")

print("\n=== MODEL MEMORY USAGE ===")

result = subprocess.run(
    ["ps", "-eo", "pid,rss,cmd", "--sort=-rss"],
    capture_output=True,
    text=True
)

for line in result.stdout.splitlines():
    if "streamlit" in line.lower() or "llama" in line.lower():
        print(line[:250])

print("\n=== APPLICATION LOGS ===")

log_path = "/content/gassafe-streamlit.log"

if os.path.exists(log_path):
    with open(log_path) as f:
        print(f.read()[-4000:])
else:
    print("Log file not found")

=== QWEN MODEL FILES ===
/content/gassafe-ai/models/qwen2.5-1.5b-instruct-q4_k_m.gguf


=== MODEL MEMORY USAGE ===
   8936 3002100 /content/gassafe-ai/.venv/bin/python -m streamlit run /content/gassafe-ai/app.py --server.address=0.0.0.0 --server.port=8501 --server.headless=true --server.enableCORS=false

=== APPLICATION LOGS ===
2026-10-15 20:58:16.953 
'server.enableCORS=false' tells Streamlit not to limit cross-origin access.
Streamlit sends the header 'Access-Control-Allow-Origin: *', except on the file
upload routes, which stay pinned to the app address and still require a valid
XSRF token.
Streamlit also accepts a WebSocket connection from any origin.
Streamlit does not set 'server.enableCORS' to 'true' for you.
'server.enableXsrfProtection=true' does not stop a cross-origin WebSocket
connection, because Streamlit does not need a valid XSRF token to open one.

To limit cross-origin access, do these steps:
  1. Set 'server.enableCORS=true'.
  2. Put the origins that you trust in 's

In [ ]:
import subprocess

code = """
from llm import generate_answer

answer = generate_answer(
    question="What is the inspection reference for Blue Pine Segment A?",
    context="Blue Pine Segment A has inspection reference BLUE-PINE-742."
)

print("QWEN GENERATED ANSWER:")
print(answer)
"""

result = subprocess.run(
    ["/content/gassafe-ai/.venv/bin/python", "-c", code],
    cwd="/content/gassafe-ai",
    capture_output=True,
    text=True,
    timeout=180
)

print("EXIT CODE:", result.returncode)
print(result.stdout[-3000:])

if result.returncode != 0:
    print("ERROR:")
    print(result.stderr[-3000:])

EXIT CODE: 0
QWEN GENERATED ANSWER:
The inspection reference for Blue Pine Segment A is BLUE-PINE-742.



In [ ]:

# =====================================================
# GasSafe AI — One-Cell Google Colab Launcher
# =====================================================

import os
import re
import time
import shutil
import subprocess
from pathlib import Path

REPO = Path("/content/gassafe-ai")
PYTHON = REPO / ".venv/bin/python"
LOG = Path("/content/gassafe-streamlit.log")
TUNNEL_LOG = Path("/content/gassafe-tunnel.log")

def run(cmd, cwd=None):
    subprocess.run(
        cmd,
        cwd=cwd,
        check=True
    )

print("STEP 1: Checking project...")

if not (REPO / "app.py").exists():
    run([
        "git", "clone",
        "https://github.com/ananthucherunthottam/gassafe-ai.git",
        str(REPO)
    ])

print("STEP 2: Checking Python environment...")

if not PYTHON.exists():
    run(["pip", "-q", "install", "uv"])
    run(["uv", "python", "install", "3.11"])
    run([
        "uv", "venv",
        "--python", "3.11",
        str(REPO / ".venv")
    ])
    run([
        "uv", "pip", "install",
        "--python", str(PYTHON),
        "-r", str(REPO / "requirements.txt")
    ])

print("STEP 3: Checking PDF knowledge base...")

check_code = """
from config import DB_DIR, COLLECTION_NAME
import chromadb

client = chromadb.PersistentClient(path=str(DB_DIR))
collection = client.get_or_create_collection(
    name=COLLECTION_NAME
)
print(collection.count())
"""

result = subprocess.run(
    [str(PYTHON), "-c", check_code],
    cwd=REPO,
    capture_output=True,
    text=True
)

try:
    chunk_count = int(result.stdout.strip())
except ValueError:
    chunk_count = 0

if result.returncode != 0 or chunk_count == 0:
    print("Building PDF index...")
    run([str(PYTHON), "ingest.py"], cwd=REPO)
else:
    print(f"PDF index ready: {chunk_count} chunks")

print("STEP 4: Starting Streamlit...")

health_url = "http://127.0.0.1:8501/_stcore/health"

import requests

def is_healthy():
    try:
        return requests.get(
            health_url,
            timeout=3
        ).status_code == 200
    except requests.RequestException:
        return False

if not is_healthy():
    streamlit_log = open(LOG, "w")

    subprocess.Popen(
        [
            str(PYTHON),
            "-m", "streamlit",
            "run", str(REPO / "app.py"),
            "--server.address=127.0.0.1",
            "--server.port=8501",
            "--server.headless=true",
            "--browser.gatherUsageStats=false"
        ],
        cwd=REPO,
        stdout=streamlit_log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    for _ in range(30):
        if is_healthy():
            break
        time.sleep(2)

if not is_healthy():
    print(LOG.read_text()[-4000:] if LOG.exists() else "")
    raise RuntimeError("Streamlit failed to start")

print("Streamlit is running!")

print("STEP 5: Starting Cloudflare Tunnel...")

CLOUDFLARED = Path("/content/cloudflared")

if not CLOUDFLARED.exists():
    run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", str(CLOUDFLARED)
    ])
    CLOUDFLARED.chmod(0o755)

# Stop any previous Quick Tunnel so the displayed
# URL corresponds to the current tunnel.
subprocess.run(
    ["pkill", "-f", "^/content/cloudflared tunnel"],
    check=False
)

tunnel_log = open(TUNNEL_LOG, "w")

tunnel = subprocess.Popen(
    [
        str(CLOUDFLARED),
        "tunnel",
        "--url", "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

url = None

for _ in range(45):
    time.sleep(2)

    logs = TUNNEL_LOG.read_text()

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        logs
    )

    if match:
        url = match.group(0)
        break

print("\n" + "=" * 45)

if url:
    print("GASSAFE AI IS READY")
    print("Open this URL:")
    print(url)
    print("\nKeep your Colab runtime connected.")
else:
    print("Tunnel failed to provide a URL.")
    print(TUNNEL_LOG.read_text()[-3000:])

print("=" * 45)

STEP 1: Checking project...
STEP 2: Checking Python environment...
STEP 3: Checking PDF knowledge base...
PDF index ready: 10 chunks
STEP 4: Starting Streamlit...
Streamlit is running!
STEP 5: Starting Cloudflare Tunnel...

GASSAFE AI IS READY
Open this URL:
https://mime-can-needs-ann.trycloudflare.com

Keep your Colab runtime connected.
